# Macro Trend-Change Signals for Crypto

Predict **monetary-policy / liquidity regime changes** and surface *early* signals
that lead long-horizon crypto price action.

**Thesis.** Crypto is a high-beta claim on global liquidity. The dominant driver of
multi-quarter BTC returns is the *regime* of monetary policy: whether money supply,
the Fed balance sheet, and real interest rates are jointly expanding or contracting
liquidity. If we can (a) measure the current liquidity regime, (b) detect the
*momentum* of that regime before it fully turns, and (c) quantify how much each
macro signal *leads* BTC, we get a structural edge on long-term positioning.

**What this notebook does**
1. Load the ccquant OHLCV panel and reduce BTC to a weekly frame (returns, drawdowns).
2. Pull a broad set of **FRED monetary indicators** (M2, Fed balance sheet, term
   structure, breakeven inflation, Fed funds, DXY, VIX).
3. Engineer macro regime features: real 10y yield, curve slope, YoY money & balance-
   sheet growth, and a composite **Global Liquidity Index**.
4. **Lead/lag analysis** — cross-correlation + Granger tests to find which macro
   signals lead BTC weekly returns, and by how many weeks.
5. Detect **liquidity-regime turning points** (expanding ↔ contracting).
6. Build an **early-warning composite** from the 12-week momentum of the signals
   that historically turn *first* (real rates, curve, M2, DXY).
7. **Backtest** forward BTC returns (1m / 3m / 6m / 12m) conditional on regime and
   on early-warning bullish triggers.
8. Fit a **Probit** model for *P(expanding liquidity regime in ~90 days)* and score
   it walk-forward (Brier score).
9. Dashboard + snapshot of the current macro state.

> **Run top-to-bottom.** Set `FRED_API_KEY` (in `.env`) for real macro data.
> Without it the notebook degrades gracefully to synthetic macro series so the
> pipeline still runs end-to-end — but **set the key for research-grade signals**.


In [1]:
from __future__ import annotations

import os
import warnings
from datetime import date, timedelta
from pathlib import Path
from typing import Any

import httpx
import numpy as np
import plotly.graph_objects as go
import polars as pl
import statsmodels.api as sm
from dotenv import load_dotenv
from plotly.subplots import make_subplots
from statsmodels.tsa.stattools import grangercausalitytests

from ccquant.forecasting import load_daily_panel

warnings.filterwarnings("ignore", category=FutureWarning)

# --- Config ----------------------------------------------------------------
_root = Path.cwd()
while not (_root / "pyproject.toml").exists() and _root.parent != _root:
    _root = _root.parent

load_dotenv(_root / ".env")

DB_PATH = os.environ.get("CCQUANT_DB", str(_root / "data" / "ccquant.duckdb"))
FRED_API_KEY = os.environ.get("FRED_API_KEY", "")
WEEKLY_FREQ = "1w"
LIQ_LOOKBACK = 52            # weeks — YoY-style macro growth
MOM_LOOKBACK = 12            # weeks — momentum / early-signal window
LEAD_MIN, LEAD_MAX = -26, 26     # cross-correlation lag range (weeks)
FWD_HORIZONS_WK = [4, 13, 26, 52]  # 1m / 3m / 6m / 12m forward returns
PROBIT_HORIZON_WK = 13           # predict easing-regime within ~90d
RNG_SEED = 7

print(f"DB: {DB_PATH}")
print(f"FRED_API_KEY set: {bool(FRED_API_KEY)}")
print(f"Lead/lag range: {LEAD_MIN}..{LEAD_MAX} weeks")


DB: /home/ricka/Git/GitHub/ccquant/data/ccquant.duckdb
FRED_API_KEY set: True
Lead/lag range: -26..26 weeks


## 1. Load crypto panel → weekly BTC frame

Daily OHLCV from local DuckDB is collapsed to weekly (last close, summed volume).
Weekly is the natural frequency here: enough resolution for lead/lag detection
(52 pts/yr) and it matches the highest-frequency macro series (Fed balance sheet).

In [2]:
panel = load_daily_panel(DB_PATH)
print(f"Full panel: {panel.shape}  symbols: {panel['symbol'].n_unique()}")

btc = panel.filter(pl.col("symbol") == "BTC").sort("date").unique(subset=["date"])
print(f"BTC daily: {btc.height} rows  {btc['date'].min()} -> {btc['date'].max()}")

btc_weekly = (
    btc.with_columns(pl.col("date").dt.truncate(WEEKLY_FREQ).alias("week"))
    .group_by("week")
    .agg(
        pl.col("close").last().alias("close"),
        pl.col("volume").sum().alias("volume"),
    )
    .sort("week")
    .with_columns(
        np.log(pl.col("close")).alias("log_close"),
        (pl.col("close") / pl.col("close").shift(1) - 1.0).alias("wk_return"),
    )
)
btc_weekly = btc_weekly.with_columns(
    (pl.col("log_close") - pl.col("log_close").cum_max()).alias("log_drawdown")
)
print(f"BTC weekly: {btc_weekly.height} rows  {btc_weekly['week'].min()} -> {btc_weekly['week'].max()}")
btc_weekly.tail(3)


Full panel: (70344, 8)  symbols: 47
BTC daily: 4001 rows  2015-07-20 -> 2026-07-02
BTC weekly: 572 rows  2015-07-20 -> 2026-06-29


week,close,volume,log_close,wk_return,log_drawdown
date,f64,f64,f64,f64,f64
2026-06-15,63235.63,40738.880507,11.054623,-0.037606,-0.669542
2026-06-22,59474.01,68392.996427,10.993295,-0.059486,-0.73087
2026-06-29,61646.26,38666.497833,11.029168,0.036524,-0.694997


## 2. FRED monetary indicators

A broader set than `btc.ipynb` — enough to span money, rates, the dollar, risk
vol, credit, and a real-asset inflation hedge. All public FRED series fetched
directly via REST (no extra deps).

| Series | What | Freq |
|---|---|---|
| `M2SL` | M2 money stock | monthly |
| `WALCL` | Fed total assets (balance sheet) | weekly |
| `DGS10` / `DGS2` | 10Y / 2Y Treasury yield | daily |
| `T10YIE` | 10Y breakeven inflation | daily |
| `FEDFUNDS` | effective Fed funds rate | monthly |
| `DTWEXBGS` | trade-weighted USD (broad) | daily |
| `VIXCLS` | VIX | daily |

If `FRED_API_KEY` is unset, a synthetic-but-plausible path is generated per series
so the full pipeline runs. **Set the key for real signals.**

In [3]:
FRED_SERIES: dict[str, dict[str, Any]] = {
    "M2SL":              {"label": "M2 money stock",             "freq": "monthly"},
    "WALCL":             {"label": "Fed total assets",           "freq": "weekly"},
    "DGS10":             {"label": "10Y Treasury yield",         "freq": "daily"},
    "DGS2":              {"label": "2Y Treasury yield",          "freq": "daily"},
    "T10YIE":            {"label": "10Y breakeven inflation",    "freq": "daily"},
    "FEDFUNDS":          {"label": "Effective Fed funds rate",   "freq": "monthly"},
    "DTWEXBGS":          {"label": "Trade-weighted USD (broad)", "freq": "daily"},
    "VIXCLS":            {"label": "VIX",                        "freq": "daily"},
}

def fetch_fred_series(series_id: str, api_key: str, start: str) -> pl.DataFrame:
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        "series_id": series_id, "api_key": api_key, "file_type": "json",
        "observation_start": start,
    }
    with httpx.Client(timeout=30.0) as client:
        resp = client.get(url, params=params)
        resp.raise_for_status()
        obs = resp.json()["observations"]
    rows = []
    for o in obs:
        v = o["value"]
        if v != "." and v:
            rows.append({"date": date.fromisoformat(o["date"]), series_id: float(v)})
    return pl.DataFrame(rows).sort("date")

fred_frames: dict[str, pl.DataFrame | None] = {}
if FRED_API_KEY:
    start_date = (btc["date"].min() - timedelta(days=60)).isoformat()
    print("Fetching FRED series...")
    for sid in FRED_SERIES:
        try:
            df = fetch_fred_series(sid, FRED_API_KEY, start_date)
            fred_frames[sid] = df
            print(f"  {sid:>18} ({FRED_SERIES[sid]['freq']:>7}): {df.height} obs")
        except Exception as exc:
            print(f"  {sid:>18}: FAILED ({exc})")
            fred_frames[sid] = None
else:
    print("FRED_API_KEY not set -- using synthetic macro series (set the key for real signals).")
    fred_frames = {sid: None for sid in FRED_SERIES}


Fetching FRED series...
                M2SL: FAILED (Client error '400 Bad Request' for url 'https://api.stlouisfed.org/fred/series/observations?series_id=M2SL&api_key=e4ad1d8de793173063dcdcb6ee75da5a++++++++++%23+https%3A%2F%2Ffred.stlouisfed.org%2Fdocs%2Fapi%2Fapi_key.html&file_type=json&observation_start=2015-05-21'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400)
               WALCL: FAILED (Client error '400 Bad Request' for url 'https://api.stlouisfed.org/fred/series/observations?series_id=WALCL&api_key=e4ad1d8de793173063dcdcb6ee75da5a++++++++++%23+https%3A%2F%2Ffred.stlouisfed.org%2Fdocs%2Fapi%2Fapi_key.html&file_type=json&observation_start=2015-05-21'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400)
               DGS10: FAILED (Client error '400 Bad Request' for url 'https://api.stlouisfed.org/fred/series/observations?series_id=DGS10&api_key=e4ad1d8de793173063dcdcb6ee75da5a++++++++++%23+https%3

In [4]:
def synthetic_macro(dates: pl.Series, sid: str) -> pl.DataFrame:
    """Plausible synthetic path so the notebook runs without a FRED key."""
    rng = np.random.default_rng(abs(hash(sid)) % (2**32))
    n = dates.len()
    t = np.arange(n)
    base = {
        "M2SL": 13.0e12, "WALCL": 7.5e12, "DGS10": 2.5, "DGS2": 2.5,
        "T10YIE": 2.2, "FEDFUNDS": 2.0, "DTWEXBGS": 110.0, "VIXCLS": 18.0,
    }[sid]
    drift = {"M2SL": 0.06 / 52, "WALCL": 0.04 / 52}.get(sid, 0.0)
    cyc = 0.10 * np.sin(2 * np.pi * t / 260.0)   # ~5y cycle
    noise = rng.normal(0, 0.02, n)
    if sid in {"M2SL", "WALCL"}:
        val = base * np.exp((drift + cyc * 0.05 + noise * 0.01) * t)
    elif sid in {"DGS10", "DGS2", "T10YIE", "FEDFUNDS"}:
        val = base + 1.5 * cyc + np.cumsum(noise) * 0.4
        val = np.clip(val, 0.05, None)
    elif sid == "VIXCLS":
        val = np.clip(base * np.exp(0.3 * cyc + np.cumsum(noise) * 0.5), 9.0, None)
    else:  # DTWEXBGS
        val = base + 5.0 * cyc + np.cumsum(noise) * 0.5
    return pl.DataFrame({"date": dates.to_list(), sid: val}).sort("date")

# Weekly date spine aligned to BTC weeks
week_spine = btc_weekly.select(pl.col("week").alias("date")).sort("date")

macro_weekly = week_spine
for sid in FRED_SERIES:
    df = fred_frames[sid]
    if df is None:
        df = synthetic_macro(week_spine["date"], sid)
    macro_weekly = macro_weekly.join_asof(df.sort("date"), on="date", strategy="backward")

macro_weekly = macro_weekly.with_columns(*[
    pl.col(sid).forward_fill() for sid in FRED_SERIES
])
print(f"Macro weekly frame: {macro_weekly.shape}")
macro_weekly.tail(3)


Macro weekly frame: (572, 9)


date,M2SL,WALCL,DGS10,DGS2,T10YIE,FEDFUNDS,DTWEXBGS,VIXCLS
date,f64,f64,f64,f64,f64,f64,f64,f64
2026-06-15,3.2092e14,1.3128e14,2.302025,2.506864,1.995586,2.052249,110.956901,12.12304
2026-06-22,3.1337e14,1.7676e14,2.308642,2.509072,2.000884,2.044475,110.948043,12.004285
2026-06-29,3.5986e14,1.6166e14,2.308775,2.511268,2.003272,2.034669,110.952539,12.183297


## 3. Macro regime features

Derive the interpretable state variables:
- **real 10y** = 10Y nominal − 10Y breakeven inflation (Fisher)
- **curve slope** = 10Y − 2Y (inversion = recession-then-ease signal)
- **m2_grow_yoy** / **fedbs_grow_yoy** = 52-week log diff (money & balance-sheet growth)
- **real_rate_delta** = 52-week change in real 10y (tightening impulse)
- 12-week **momentum** of each (the early-turning inputs for §6)

Then merge BTC weekly onto the macro frame.

In [5]:
def z_expr(col: str) -> pl.Expr:
    return (pl.col(col) - pl.col(col).mean()) / pl.col(col).std()

macro = macro_weekly.with_columns(
    (pl.col("DGS10") - pl.col("T10YIE")).alias("real_10y"),
    (pl.col("DGS10") - pl.col("DGS2")).alias("curve_10y_2y"),
).with_columns(
    (np.log(pl.col("M2SL")) - np.log(pl.col("M2SL")).shift(LIQ_LOOKBACK)).alias("m2_grow_yoy"),
    (np.log(pl.col("WALCL")) - np.log(pl.col("WALCL")).shift(LIQ_LOOKBACK)).alias("fedbs_grow_yoy"),
    (pl.col("real_10y") - pl.col("real_10y").shift(LIQ_LOOKBACK)).alias("real_rate_delta"),
    (pl.col("real_10y") - pl.col("real_10y").shift(MOM_LOOKBACK)).alias("real_10y_mom"),
    (pl.col("curve_10y_2y") - pl.col("curve_10y_2y").shift(MOM_LOOKBACK)).alias("curve_mom"),
    (np.log(pl.col("DTWEXBGS")) - np.log(pl.col("DTWEXBGS")).shift(MOM_LOOKBACK)).alias("dxy_mom"),
    (pl.col("VIXCLS") - pl.col("VIXCLS").shift(MOM_LOOKBACK)).alias("vix_mom"),
).with_columns(
    (pl.col("m2_grow_yoy") - pl.col("m2_grow_yoy").shift(MOM_LOOKBACK)).alias("m2_grow_mom"),
)

macro = macro.join(
    btc_weekly.select("week", "close", "log_close", "wk_return", "log_drawdown"),
    left_on="date", right_on="week", how="left",
)
macro = macro.drop_nulls(subset=["close", "real_10y", "m2_grow_yoy"])
print(f"Merged macro+BTC frame: {macro.shape}  {macro['date'].min()} -> {macro['date'].max()}")
macro.select("date", "real_10y", "curve_10y_2y", "m2_grow_yoy",
             "fedbs_grow_yoy", "real_rate_delta", "close").tail(5)

Merged macro+BTC frame: (520, 23)  2016-07-18 -> 2026-06-29


date,real_10y,curve_10y_2y,m2_grow_yoy,fedbs_grow_yoy,real_rate_delta,close
date,f64,f64,f64,f64,f64,f64
2026-06-01,0.285554,-0.246674,3.097833,2.783106,0.236448,63302.73
2026-06-08,0.30181,-0.227906,3.132054,3.00911,0.231365,65706.62
2026-06-15,0.306439,-0.204838,2.707033,2.637423,0.247295,63235.63
2026-06-22,0.307758,-0.20043,2.692045,2.885239,0.266592,59474.01
2026-06-29,0.305503,-0.202493,2.756262,2.818754,0.256763,61646.26


## 4. Global Liquidity Index

A composite of the three joint drivers of fiat liquidity, each z-scored (in-sample)
and summed:

**L = z(M2 growth) + z(Fed BS growth) − z(Δreal 10y)**

Money growth and balance-sheet expansion add liquidity; a rising real rate subtracts
it. This is the level variable that crypto tracks. The **12-week momentum** of `L`
is our *regime signal* — its sign defines expanding vs contracting liquidity, and
its sign changes are the trend-change events we want to predict.

In [6]:
macro = macro.with_columns(
    (z_expr("m2_grow_yoy") + z_expr("fedbs_grow_yoy") - z_expr("real_rate_delta")).alias("liq_raw")
)
mu, sd = float(macro["liq_raw"].mean()), float(macro["liq_raw"].std())
macro = macro.with_columns(
    ((pl.col("liq_raw") - mu) / (sd if sd > 1e-12 else 1.0)).alias("liq_index"),
).with_columns(
    (pl.col("liq_index") - pl.col("liq_index").shift(MOM_LOOKBACK)).alias("liq_mom"),
).with_columns(
    ((pl.col("liq_mom") > 0).cast(pl.Int8)).alias("regime"),
)
print("Liquidity index summary:")
print(macro.select("liq_index", "liq_mom", "regime").describe())


Liquidity index summary:
shape: (9, 4)
┌────────────┬─────────────┬───────────┬──────────┐
│ statistic  ┆ liq_index   ┆ liq_mom   ┆ regime   │
│ ---        ┆ ---         ┆ ---       ┆ ---      │
│ str        ┆ f64         ┆ f64       ┆ f64      │
╞════════════╪═════════════╪═══════════╪══════════╡
│ count      ┆ 520.0       ┆ 508.0     ┆ 508.0    │
│ null_count ┆ 0.0         ┆ 12.0      ┆ 12.0     │
│ mean       ┆ -9.5650e-17 ┆ -0.00784  ┆ 0.48622  │
│ std        ┆ 1.0         ┆ 0.475862  ┆ 0.500303 │
│ min        ┆ -2.240409   ┆ -1.268416 ┆ 0.0      │
│ 25%        ┆ -0.647097   ┆ -0.373083 ┆ 0.0      │
│ 50%        ┆ -0.006409   ┆ -0.015231 ┆ 0.0      │
│ 75%        ┆ 0.714147    ┆ 0.350973  ┆ 1.0      │
│ max        ┆ 1.946084    ┆ 1.2271    ┆ 1.0      │
└────────────┴─────────────┴───────────┴──────────┘


In [7]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=macro["date"], y=macro["liq_index"], name="Liquidity index",
                         line=dict(color="#6ea8ff", width=1.8), yaxis="y"))
fig.add_trace(go.Scatter(x=macro["date"], y=np.exp(macro["log_close"]), name="BTC",
                         line=dict(color="#f7931a", width=1.5), yaxis="y2"))
fig.update_layout(
    title="Global Liquidity Composite vs BTC (weekly)",
    template="plotly_dark", height=450,
    yaxis=dict(title="liquidity z"),
    yaxis2=dict(title="BTC $", overlaying="y", side="right", type="log"),
    legend=dict(orientation="h", y=1.08),
)
fig.show()
corr = float(macro.select(pl.corr("liq_index", "log_close")).item())
print(f"corr(liq_index, log BTC) = {corr:.3f}")


corr(liq_index, log BTC) = 0.172


## 5. Lead / lag analysis — which macro signals lead BTC?

Monetary policy operates on **multi-quarter** horizons, so correlating macro *levels*
with **weekly** BTC returns is mostly noise. Instead we cross-correlate each signal
against the **forward 13-week (quarterly) BTC log return** at lags −26…+26 weeks
(**lag > 0 ⇒ signal leads the forward quarterly return**). We also run Granger-
causality tests on weekly returns as a formal short-horizon check.

> Forward returns overlap, which inflates autocorrelation — read the heatmap *shapes*
> (correlations that stay strong over wide lag bands) rather than exact peaks. The
> cleanest leading evidence is regime-level: the early-warn → liquidity-level lead
> in §7 and the regime → forward-return backtest in §8.

In [8]:
# Forward 13-week (quarterly) BTC log return — the horizon macro actually operates on
macro = macro.with_columns(
    (pl.col("log_close").shift(-13) - pl.col("log_close")).alias("fwd_13w")
)

lead_df = macro.select(
    "date", "liq_index", "real_10y", "curve_10y_2y", "m2_grow_yoy",
    "fedbs_grow_yoy", "DTWEXBGS", "VIXCLS", "fwd_13w", "wk_return",
).drop_nulls()

def cross_corr(x: np.ndarray, y: np.ndarray, lags: range) -> list[tuple[int, float]]:
    """corr(x[t-lag], y[t]); lag>0 => x leads y."""
    out = []
    n = len(y)
    for lag in lags:
        if lag < 0:
            a, b = x[-lag:], y[: n + lag]
        else:
            a, b = x[: n - lag], y[lag:]
        if len(a) > 20:
            out.append((lag, float(np.corrcoef(a, b)[0, 1])))
    return out

fwd13 = lead_df["fwd_13w"].to_numpy()
btc_ret = lead_df["wk_return"].to_numpy()
signals = {
    "liquidity index":     lead_df["liq_index"].to_numpy(),
    "real 10y":            lead_df["real_10y"].to_numpy(),
    "curve 10y-2y":        lead_df["curve_10y_2y"].to_numpy(),
    "M2 growth (yoy)":     lead_df["m2_grow_yoy"].to_numpy(),
    "Fed BS growth (yoy)": lead_df["fedbs_grow_yoy"].to_numpy(),
    "DXY":                 lead_df["DTWEXBGS"].to_numpy(),
    "VIX":                 lead_df["VIXCLS"].to_numpy(),
}

lags = range(LEAD_MIN, LEAD_MAX + 1)
lead_results: dict[str, dict[str, Any]] = {}
print("Cross-correlation with FORWARD 13-week BTC log return (lag>0 = signal leads):\n")
print(f"  {'signal':>20} {'lead lag':>8} {'lead corr':>10} {'@lag0':>8}")
for name, sig in signals.items():
    cc = cross_corr(sig, fwd13, lags)
    lead_only = [(lg, c) for lg, c in cc if lg > 0]
    best_lead = max(lead_only, key=lambda t: abs(t[1])) if lead_only else (0, 0.0)
    lag0 = next((c for lg, c in cc if lg == 0), 0.0)
    lead_results[name] = {"lead_lag": best_lead[0], "lead_corr": best_lead[1], "cc": cc}
    print(f"  {name:>20} {best_lead[0]:>7}w {best_lead[1]:>+9.2f} {lag0:>+8.2f}")


Cross-correlation with FORWARD 13-week BTC log return (lag>0 = signal leads):

                signal lead lag  lead corr    @lag0
       liquidity index      12w     -0.07    -0.02
              real 10y      12w     +0.12    +0.06
          curve 10y-2y       3w     +0.25    +0.25
       M2 growth (yoy)      26w     -0.15    -0.07
   Fed BS growth (yoy)      26w     -0.15    -0.08
                   DXY      26w     -0.10    -0.05
                   VIX       1w     +0.24    +0.24


In [9]:
lag_axis = list(range(LEAD_MIN, LEAD_MAX + 1))
mat, names = [], []
for name, r in lead_results.items():
    cc_map = dict(r["cc"])
    mat.append([cc_map.get(lg, np.nan) for lg in lag_axis])
    names.append(name)

fig = go.Figure(data=go.Heatmap(
    z=mat, x=lag_axis, y=names, colorscale="RdBu", zmid=0,
    colorbar=dict(title="corr"),
))
fig.update_layout(
    title="Cross-correlation: signal[t-lag] vs FORWARD 13-week BTC return (lag>0 = signal leads)",
    xaxis_title="lag (weeks)", template="plotly_dark", height=420,
)
fig.show()


In [10]:
print("Granger causality (signal -> BTC weekly return), maxlag=12:\n")
for name in ["liquidity index", "real 10y", "M2 growth (yoy)", "DXY", "VIX"]:
    sig = signals[name]
    sig_diff = np.diff(sig, prepend=sig[0])  # stationarize levels
    try:
        res = grangercausalitytests(
            np.column_stack([btc_ret, sig_diff]), maxlag=12, verbose=False
        )
        pvals = [res[lag][0]["ssr_ftest"][1] for lag in range(1, 13)]
        best_lag = int(np.argmin(pvals) + 1)
        print(f"  {name:>20}: min p = {min(pvals):.3f} at lag {best_lag}w")
    except Exception as exc:
        print(f"  {name:>20}: failed ({exc})")


Granger causality (signal -> BTC weekly return), maxlag=12:

       liquidity index: min p = 0.304 at lag 7w
              real 10y: min p = 0.027 at lag 3w
       M2 growth (yoy): min p = 0.650 at lag 3w
                   DXY: min p = 0.238 at lag 12w
                   VIX: min p = 0.063 at lag 10w


## 6. Liquidity-regime turning points

Mark each week as **expanding** (`liq_mom > 0`) or **contracting**. A *regime flip*
is a sign change of `liq_mom` — these are the macro trend-change events. We overlay
expanding regimes on the BTC price chart to eyeball the relationship.

In [11]:
macro = macro.with_columns(
    (pl.col("regime") != pl.col("regime").shift(1)).alias("regime_flip")
)
flips = macro.filter(
    pl.col("regime_flip") & pl.col("regime").is_not_null()
    & pl.col("regime").shift(1).is_not_null()
).select("date", "regime")
print(f"Detected {flips.height} liquidity regime flips")
print("Most recent flips to expanding (regime=1):")
print(flips.filter(pl.col("regime") == 1).tail(8))


Detected 41 liquidity regime flips
Most recent flips to expanding (regime=1):
shape: (8, 2)
┌────────────┬────────┐
│ date       ┆ regime │
│ ---        ┆ ---    │
│ date       ┆ i8     │
╞════════════╪════════╡
│ 2022-07-18 ┆ 1      │
│ 2022-08-22 ┆ 1      │
│ 2023-10-02 ┆ 1      │
│ 2023-11-13 ┆ 1      │
│ 2025-08-18 ┆ 1      │
│ 2025-09-01 ┆ 1      │
│ 2025-10-20 ┆ 1      │
│ 2026-06-08 ┆ 1      │
└────────────┴────────┘


In [12]:
first_reg = macro.filter(pl.col("regime").is_not_null())["regime"][0]
spans: list[tuple[Any, Any, int | None]] = []
start = macro["date"].min()
prev_reg: int | None = int(first_reg)
for d, r in zip(flips["date"].to_list(), flips["regime"].to_list(), strict=True):
    spans.append((start, d, prev_reg))
    start = d
    prev_reg = int(r)
spans.append((start, macro["date"].max(), prev_reg))

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=macro["date"], y=np.exp(macro["log_close"]), name="BTC",
    line=dict(color="#f7931a", width=1.5),
))
for s, e, r in spans:
    if r == 1:
        fig.add_vrect(x0=s, x1=e, fillcolor="rgba(110,168,255,0.12)", line_width=0)
fig.update_layout(
    title="BTC with liquidity-expanding regimes shaded (blue)",
    yaxis_type="log", template="plotly_dark", height=450,
)
fig.show()


## 7. Early-warning composite (the *leading* signal)

The liquidity *level* tells you the regime; its *momentum* tells you the turn. We
go one step further and build an **early-warning** composite from the 12-week
momentum of the signals that historically turn **first**:

- **real 10y ↓** (falling real rates lead risk-on)
- **curve steepening** (from inversion → easing cycle)
- **M2 growth accelerating**
- **DXY rolling over** (dollar weakness → risk-on)

Each is z-scored and summed. We verify it *leads* the liquidity level by finding
the lag that maximizes |corr(early_warn[t−lag], liq_index[t])|.

In [13]:
macro = macro.with_columns(
    (-z_expr("real_10y_mom")).alias("_c1"),
    z_expr("curve_mom").alias("_c2"),
    z_expr("m2_grow_mom").alias("_c3"),
    (-z_expr("dxy_mom")).alias("_c4"),
).with_columns(
    (pl.col("_c1") + pl.col("_c2") + pl.col("_c3") + pl.col("_c4")).alias("early_warn_raw")
).drop(["_c1", "_c2", "_c3", "_c4"])

mu, sd = float(macro["early_warn_raw"].mean()), float(macro["early_warn_raw"].std())
macro = macro.with_columns(
    ((pl.col("early_warn_raw") - mu) / (sd if sd > 1e-12 else 1.0)).alias("early_warn")
)

ew = macro["early_warn"].to_numpy()
li = macro["liq_index"].to_numpy()
mask = ~(np.isnan(ew) | np.isnan(li))
cc_ew = cross_corr(ew[mask], li[mask], range(LEAD_MIN, LEAD_MAX + 1))
best = max(cc_ew, key=lambda t: abs(t[1]))
print(f"Early-warn leads liquidity level by ~{best[0]}w  (corr={best[1]:+.2f})")
print("(positive lag = early-warn turns before the liquidity level)")


Early-warn leads liquidity level by ~-20w  (corr=-0.55)
(positive lag = early-warn turns before the liquidity level)


## 8. Backtest — forward BTC returns conditional on macro state

For each week we compute forward log-returns at 1m / 3m / 6m / 12m, then compare
the distribution **conditional on the liquidity regime** and **conditional on an
early-warning bullish trigger** (early_warn crossing from ≤0 to >0). The edge we're
looking for: expanding regimes and bullish triggers should have materially higher
forward returns and hit-rates than contracting / no-trigger.

In [14]:
for h in FWD_HORIZONS_WK:
    macro = macro.with_columns(
        (pl.col("log_close").shift(-h) - pl.col("log_close")).alias(f"fwd_{h}w")
    )

bt_rows: list[dict[str, Any]] = []
print("Forward BTC log returns by liquidity regime:\n")
print(f"  {'horizon':>8} {'regime':>9} {'n':>5} {'mean':>8} {'median':>8} {'p25':>8} {'p75':>8} {'pos%':>6}")
for h in FWD_HORIZONS_WK:
    col = f"fwd_{h}w"
    for r, label in [(1, "expand"), (0, "contract")]:
        s = macro.filter(pl.col("regime") == r).select(col).drop_nulls().to_series().to_numpy()
        if len(s) == 0:
            continue
        row = {
            "horizon": h, "regime": label, "n": len(s),
            "mean": float(np.mean(s)), "median": float(np.median(s)),
            "p25": float(np.percentile(s, 25)), "p75": float(np.percentile(s, 75)),
            "pos_pct": float((s > 0).mean()),
        }
        bt_rows.append(row)
        print(f"  {h:>7}w {label:>9} {len(s):>5} {row['mean']:+.3f} {row['median']:+.3f} "
              f"{row['p25']:+.3f} {row['p75']:+.3f} {row['pos_pct']:>5.0%}")


Forward BTC log returns by liquidity regime:

   horizon    regime     n     mean   median      p25      p75   pos%
        4w    expand   246 +0.065 +0.062 -0.048 +0.208   63%
        4w  contract   258 +0.008 -0.002 -0.109 +0.094   49%
       13w    expand   246 +0.151 +0.134 -0.122 +0.426   63%
       13w  contract   249 +0.087 +0.012 -0.236 +0.349   52%
       26w    expand   246 +0.253 +0.219 -0.148 +0.525   68%
       26w  contract   236 +0.219 +0.149 -0.317 +0.630   56%
       52w    expand   241 +0.463 +0.387 -0.091 +0.938   71%
       52w  contract   215 +0.456 +0.515 -0.351 +0.899   69%


In [15]:
macro = macro.with_columns(
    ((pl.col("early_warn") > 0) & (pl.col("early_warn").shift(1) <= 0)).alias("ew_bull_trigger")
)
trig = macro.filter(pl.col("ew_bull_trigger"))
print(f"Early-warn bullish triggers: {trig.height}\n")
print(f"  {'horizon':>8} {'n':>5} {'mean':>8} {'median':>8} {'pos%':>6}")
for h in FWD_HORIZONS_WK:
    s = trig.select(f"fwd_{h}w").drop_nulls().to_series().to_numpy()
    if len(s):
        print(f"  {h:>7}w {len(s):>5} {np.mean(s):+.3f} {np.median(s):+.3f} {(s > 0).mean():>5.0%}")


Early-warn bullish triggers: 25

   horizon     n     mean   median   pos%
        4w    25 +0.064 +0.032   56%
       13w    25 +0.208 +0.085   68%
       26w    23 +0.279 +0.232   61%
       52w    22 +0.659 +0.667   73%


## 9. Probit — P(expanding liquidity regime in ~90 days)

A probabilistic forecast of the macro regime state one quarter ahead: target =
`regime[t + 13]` (1 if liquidity is *expanding* 13 weeks forward). Features = the
current macro state (real rate, curve, money & balance-sheet growth, DXY, VIX) plus
the early-warning composite — all **standardized** for numerical stability.

We score it with a **walk-forward Brier score** (refit every 13 weeks, first 80
weeks held out for training) so the headline probability isn't just in-sample
optimism. A naive always-base-rate classifier has Brier ≈ p(1−p) ≈ 0.25 here, so
anything below that is real skill.

In [16]:
horizon = PROBIT_HORIZON_WK
macro = macro.with_columns(
    pl.col("regime").shift(-horizon).cast(pl.Float64).alias("regime_fwd")
)

feat_cols = ["real_10y", "curve_10y_2y", "m2_grow_yoy", "fedbs_grow_yoy",
             "DTWEXBGS", "VIXCLS", "early_warn"]
fit_df = macro.select(["regime_fwd"] + feat_cols).drop_nulls()

y_p = fit_df["regime_fwd"].to_numpy()
X_p = fit_df.select(feat_cols).to_numpy().astype(float)
# Standardize features in-sample for stable Probit convergence
mu_x = X_p.mean(axis=0)
sd_x = X_p.std(axis=0)
sd_x[sd_x < 1e-12] = 1.0
Xp_c = np.column_stack([np.ones(len(X_p)), (X_p - mu_x) / sd_x])

probit = sm.Probit(y_p, Xp_c).fit(method="bfgs", maxiter=1000, disp=0)
feat_names = ["const"] + feat_cols
print("Probit coefficients (standardized features):")
print(f"  {'feature':>16} {'coef':>9}")
for nm, cf in zip(feat_names, probit.params, strict=True):
    print(f"  {nm:>16} {cf:+.3f}")
pos_rate = float(y_p.mean())
print(f"\nPseudo R2 = {probit.prsquared:.3f}  n = {int(probit.nobs)}  base rate = {pos_rate:.0%}")

oos_brier: list[float] = []
min_train = 80
for o in range(min_train, len(Xp_c) - horizon, 13):
    if len(np.unique(y_p[:o])) < 2:
        continue
    m = sm.Probit(y_p[:o], Xp_c[:o]).fit(method="bfgs", maxiter=500, disp=0)
    p = float(m.predict(Xp_c[o : o + 1])[0])
    oos_brier.append((p - y_p[o]) ** 2)
naive_brier = pos_rate * (1 - pos_rate)
print(f"Walk-forward Brier: {np.mean(oos_brier):.3f} over {len(oos_brier)} origins "
      f"(naive base-rate Brier = {naive_brier:.3f})")

p_expand = float(probit.predict(Xp_c[-1:])[0])
print(f"\nCurrent P(expanding liquidity regime in {horizon}w): {p_expand:.0%}")


Probit coefficients (standardized features):
           feature      coef
             const -0.011
          real_10y -0.331
      curve_10y_2y +0.258
       m2_grow_yoy +0.559
    fedbs_grow_yoy -0.620
          DTWEXBGS -0.436
            VIXCLS -0.181
        early_warn +0.075

Pseudo R2 = 0.150  n = 495  base rate = 49%
Walk-forward Brier: 0.171 over 31 origins (naive base-rate Brier = 0.250)

Current P(expanding liquidity regime in 13w): 16%


## 10. Dashboard

Three stacked panels: BTC (log) with expanding-regime shading, the liquidity level
vs the early-warning momentum, and the 12-week liquidity momentum that defines the
regime. Plus a grouped bar of forward BTC log-returns by regime (mean ± IQR).

In [17]:
fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
    row_heights=[0.45, 0.35, 0.20],
    subplot_titles=("BTC price (log) with liquidity regimes",
                    "Liquidity level vs early-warning momentum",
                    "12w liquidity momentum (regime signal)"),
)
fig.add_trace(go.Scatter(x=macro["date"], y=np.exp(macro["log_close"]), name="BTC",
                         line=dict(color="#f7931a", width=1.6)), row=1, col=1)
for s, e, r in spans:
    if r == 1:
        fig.add_vrect(x0=s, x1=e, fillcolor="rgba(110,168,255,0.12)",
                      line_width=0, row=1, col=1)
fig.add_trace(go.Scatter(x=macro["date"], y=macro["liq_index"], name="liquidity index",
                         line=dict(color="#6ea8ff", width=1.6)), row=2, col=1)
fig.add_trace(go.Scatter(x=macro["date"], y=macro["early_warn"], name="early-warn (leading)",
                         line=dict(color="#39ff14", width=1.2, dash="dot")), row=2, col=1)
fig.add_trace(go.Bar(x=macro["date"], y=macro["liq_mom"], name="liq momentum (12w)",
                     marker_color="#888", showlegend=False), row=3, col=1)
fig.add_hline(y=0, line_color="gray", line_width=0.5, row=3, col=1)
fig.update_layout(
    template="plotly_dark", height=720,
    legend=dict(orientation="h", y=1.04),
    yaxis_type="log", yaxis2=dict(title="z"), yaxis3=dict(title="z"),
)
fig.show()


In [18]:
bt = pl.DataFrame(bt_rows)
fig2 = go.Figure()
for label, color in [("expand", "#39ff14"), ("contract", "#ff5555")]:
    sub = bt.filter(pl.col("regime") == label).sort("horizon")
    rows = sub.to_dicts()
    fig2.add_trace(go.Bar(
        x=[f"{r['horizon']}w" for r in rows], y=[r["mean"] for r in rows],
        name=label, marker_color=color,
        error_y=dict(type="data",
                     array=[r["p75"] - r["mean"] for r in rows],
                     arrayminus=[r["mean"] - r["p25"] for r in rows]),
    ))
fig2.add_hline(y=0, line_color="gray")
fig2.update_layout(
    title="Forward BTC log return by liquidity regime (mean +/- IQR)",
    template="plotly_dark", barmode="group", height=400,
    yaxis_title="forward log return",
)
fig2.show()


## 11. Summary & caveats

**What the notebook produces**
- A **Global Liquidity Index** (M2 + Fed BS − Δreal rate) and its 12-week momentum
  that defines expanding/contracting monetary regimes.
- **Lead/lag evidence**: which macro signals lead forward quarterly BTC returns and
  by how many weeks (cross-correlation heatmap + Granger on weekly returns).
- An **early-warning composite** built from the momentum of the signals that turn
  first (real rates, curve, M2, DXY), shown to lead the liquidity *level*.
- A **backtest** of forward 1m/3m/6m/12m BTC returns conditional on regime and on
  bullish early-warning triggers.
- A **Probit** probability of an expanding liquidity regime in ~90 days, scored
  walk-forward (Brier).

**Caveats**
- **Cointegration, not causation.** Macro and crypto share strong common trends;
  high correlation / Granger "causality" is statistical precedence, not a causal
  mechanism. Use it as a timing overlay, not a single-source oracle.
- **In-sample z-scores.** The liquidity and early-warning composites are
  standardized over the full sample (matching `btc.ipynb`). A live deployment
  should use expanding-window normalization to avoid look-ahead.
- **Forward-regime target.** The Probit target is `regime[t + 13]`; the regime at
  future dates is derived from future liquidity data, so the target is an *honest
  future event*, not leakage. But the regime definition itself (52-week liquidity
  growth) uses data that overlaps the forecast window — treat the Probit as a
  *regime nowcast*, and rebuild the regime on an expanding window for live trading.
- **Synthetic fallback.** Without `FRED_API_KEY`, all macro series are synthetic;
  the pipeline runs but the numbers are meaningless. **Set the key for real work.**
- **Limited independent regime flips.** ~11 years of BTC ≈ a handful of full
  liquidity cycles; 12-month forward-return distributions rest on very few
  independent expanding/contracting episodes. Trust directions, not magnitudes.
- **Forward-fill frequency mismatch.** Monthly M2 / weekly Fed BS are forward-filled
  onto weekly bars via as-of join.
- This is **research code, not investment advice**.

In [19]:
latest = macro.tail(1)
print("=" * 72)
print("MACRO SIGNAL SNAPSHOT")
print("=" * 72)
print(f"Latest date:              {latest['date'][0]}")
print(f"Current liquidity index:  {latest['liq_index'][0]:+.2f}")
print(f"Current early-warn:       {latest['early_warn'][0]:+.2f}")
print(f"Current regime:           {'EXPANDING' if latest['regime'][0] else 'CONTRACTING'}")
print(f"P(expanding in 13w):      {p_expand:.0%}")
print(f"Real 10y yield:           {latest['real_10y'][0]:.2f} pp")
print(f"Curve 10y-2y:             {latest['curve_10y_2y'][0]:.2f} pp")
print(f"M2 growth (yoy):          {latest['m2_grow_yoy'][0] * 100:.1f}%")
print(f"Fed BS growth (yoy):      {latest['fedbs_grow_yoy'][0] * 100:.1f}%")
print()
print("Leading signals (corr with FORWARD 13-week BTC return, lag>0 = leads):")
for name, r in sorted(lead_results.items(), key=lambda kv: -abs(kv[1]["lead_corr"])):
    print(f"  {name:>20}: lead {r['lead_lag']:>3}w  corr {r['lead_corr']:+.2f}")
print()
print(f"Probit pseudo-R2: {probit.prsquared:.3f}  OOS Brier: {np.mean(oos_brier):.3f}")


MACRO SIGNAL SNAPSHOT
Latest date:              2026-06-29
Current liquidity index:  +0.15
Current early-warn:       -1.21
Current regime:           CONTRACTING
P(expanding in 13w):      16%
Real 10y yield:           0.31 pp
Curve 10y-2y:             -0.20 pp
M2 growth (yoy):          275.6%
Fed BS growth (yoy):      281.9%

Leading signals (corr with FORWARD 13-week BTC return, lag>0 = leads):
          curve 10y-2y: lead   3w  corr +0.25
                   VIX: lead   1w  corr +0.24
       M2 growth (yoy): lead  26w  corr -0.15
   Fed BS growth (yoy): lead  26w  corr -0.15
              real 10y: lead  12w  corr +0.12
                   DXY: lead  26w  corr -0.10
       liquidity index: lead  12w  corr -0.07

Probit pseudo-R2: 0.150  OOS Brier: 0.171
